<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/fine_tuning_Ed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q typing transformers datasets python_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
# imports

import os
import random
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict

#from items import Item

import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import numpy as np
import pickle

In [4]:
from typing import Optional
from transformers import AutoTokenizer
import re
from datasets import load_dataset

In [130]:
data = load_dataset("pgurazada1/amazon_india_products")["train"]

In [131]:
# Investigate a particular datapoint
datapoint = data[2]

In [132]:
datapoint

{'Uniq Id': '41654633cce38c8650690f6dbac01fd3',
 'Crawl Timestamp': '2019-10-30 09:53:23 +0000',
 'Category': 'Skin Care',
 'Product Title': ' Generic 1 Pc brand snail eye cream remove dark circle eye lifting instant ageless snail cream for eye care anti wrinkle eyes cream 20g ',
 'Product Description': 'Use: eye, item type: cream, net wt: 20g, gzzz: ygzwbz, model number: lkwnys, gender: female, certification: gzzz, feature: moisturizing, dark circle, anti-puffiness, anti-aging, brand name: laikou, certificate number: 2014028630, ingredient: cream, country/region of manufacture: china, eye care features: eye cream, product name: snail extract cream, applicable to the crowd: general, specifications: normal specifications, origin: shantou, guangdong province, unit type: piece, package weight: ,package size:',
 'Brand': 'Generic',
 'Pack Size Or Quantity': None,
 'Mrp': '1824.00',
 'Price': '1042.00',
 'Site Name': 'Amazon In',
 'Offers': '42.87%',
 'Combo Offers': None,
 'Stock Availibil

In [133]:
# How many have prices?

prices = 0
for datapoint in data:
    try:
        price = float(datapoint["Mrp"] or 0.0)
        if price > 0.0:
            prices += 1
    except ValueError as e:
        pass

print(f"There are {prices:,} with prices which is {prices/len(data)*100:,.1f}%")

There are 29,240 with prices which is 97.5%


In [134]:
import pandas as pd
df  = pd.DataFrame(data)

# Round the Mrp column to avoid data type issues downstream



In [135]:
df.iloc[58]

,58
Uniq Id,7602f259a7d1d2c35904f1aca7f43923
Crawl Timestamp,2019-10-30 02:32:10 +0000
Category,Bath & Shower
Product Title,Donna Karan Cashmere Mist Hand Treatment Crem...
Product Description,Cashmere mist by donna Karan for women - 2.5 o...
Brand,Donna Karan
Pack Size Or Quantity,None
Mrp,7034.50
Price,7034.50
Site Name,Amazon In


In [136]:
#Drop the rows where the price is none and then delete rows where price is 0.0

df = df.dropna(subset=['Mrp'])



In [75]:
#Get the raw df to see th diferent values of the price

df.to_csv('raw_df.csv')

In [137]:
df.dtypes

,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,object
Price,object
Site Name,object


In [142]:
# CHange the data tyoe of the column to float
df['Mrp'].fillna('0', inplace=True)
import re

def clean_price(price_str):
  str_price = str(price_str)
  """Cleans a price string by removing extra decimal points."""
  if str_price.count('.') ==1:
    return str_price.replace('.', '')
  elif str_price.count('.') ==2:
    cleaned_price = re.sub(r"\.", "", str_price.strip(), count=1)
    return cleaned_price
  else:
    return str_price

In [143]:
# added code to colab

In [145]:
# apply the clean price function

df['Mrp'] = df['Mrp'].apply(lambda x : clean_price(x))
df['dec_count'] = df['Mrp'].apply(lambda x:[0 if x is None else x.count('.')][0])


In [146]:
df.groupby('dec_count')['dec_count'].sum()

,dec_count
dec_count,
0,0


In [111]:
df.dtypes

,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,object
Price,object
Site Name,object


Check before we use the Item class if the data frame consists of product description, title and Mrp

In [147]:
#df[df['Product Description'].isna()==True] #1907 rows

#df[df['Product Title'].isna()==True] #0 rows
#df[df['Mrp']==0.0] # 0 rows

In [148]:
df.shape

(29301, 16)

In [149]:
df[df['Product Description'].isna()==True]

,Uniq Id,Crawl Timestamp,Category,Product Title,Product Description,Brand,Pack Size Or Quantity,Mrp,Price,Site Name,Offers,Combo Offers,Stock Availibility,Product Asin,Image Urls,dec_count
41,fb5f58e96d3ad0f614c3205d484236e6,2019-10-31 08:40:01 +0000,Grocery & Gourmet Foods,Wonderland Foods Roasted & Salted Cashews 200...,None,Wonderland,200 Grams,360000,311.00,Amazon In,13.61%,None,YES,B01M7RNC92,https://images-na.ssl-images-amazon.com/images...,0
42,5782bea6213fc3998f3be9d0574e34a6,2019-10-30 05:33:56 +0000,Skin Care,"Bekind Rose Flower Powder, 40 g (Pack of 4)",None,Bekind,None,492000,292.00,Amazon In,40.65%,None,YES,B075R7SD9F,https://images-na.ssl-images-amazon.com/images...,0
47,33df63d0abc9cb54b4d659d5b7b02791,2019-10-30 05:55:53 +0000,Bath & Shower,FidgetGear Wooden Christmas Series Hanging Pe...,None,FidgetGear,None,645980,514.00,Amazon In,20.43%,None,YES,B07YPLR5QP,https://images-na.ssl-images-amazon.com/images...,0
49,d097e144c3d658845ff5b4bece86530f,2019-10-30 04:12:30 +0000,Skin Care,3nh Gel SPA Socks Moisturizing Whitening Exfo...,None,3nh,None,1738000,993.00,Amazon In,42.87%,None,YES,B07Y3LQZH8,https://images-na.ssl-images-amazon.com/images...,0
112,d95cfa2c31127e4388bbd4c3e623137d,2019-10-30 11:38:40 +0000,Skin Care,KAZIMA ALOE VERA COLD CREAM (100g) with Almon...,None,KAZIMA,99.8 g,235000,235.00,Amazon In,0%,None,YES,B07L1LWC76,https://images-na.ssl-images-amazon.com/images...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29961,ef7752a0b25222447f13480e351b162a,2019-10-30 04:24:19 +0000,Skin Care,Madofy Grade a thanaka (tanaka) powder 50gm k...,None,MADOFY,49.9 g,1479000,1099.00,Amazon In,25.69%,None,YES,B07PXXBQYQ,https://images-na.ssl-images-amazon.com/images...,0
29968,c7153c3b35f6654c21a86c33a8b5cf0b,2019-10-31 04:22:08 +0000,Detergents & Dishwash,"EPIC Mountain Blue Laundry Detergent Liquid,D...",None,Epic,None,250000,169.00,Amazon In,32.4%,None,YES,B07NHJJ2VC,https://images-na.ssl-images-amazon.com/images...,0
29994,390d24de64395cc8a66c8e5d43c03c0a,2019-10-29 03:28:49 +0000,Fragrance,Rose Water & saffron Water Combo,None,Aadi Vedaa,None,599000,599.00,Amazon In,0%,None,YES,B076GM24HR,https://images-na.ssl-images-amazon.com/images...,0
29997,6038643bc2adfd0228acf4eeb8c4a4fc,2019-10-30 22:54:39 +0000,Skin Care,Elancyl Slim Design Flat Stomach 150ml,None,None,None,7287000,7287.00,Amazon In,0%,None,YES,B078S7K42M,https://images-na.ssl-images-amazon.com/images...,0


In [150]:
final_df = df[~df['Product Description'].isna()]

In [151]:
final_df.shape

(27394, 16)

In [152]:
final_df.columns

Index(['Uniq Id', 'Crawl Timestamp', 'Category', 'Product Title',
       'Product Description', 'Brand', 'Pack Size Or Quantity', 'Mrp', 'Price',
       'Site Name', 'Offers', 'Combo Offers', 'Stock Availibility',
       'Product Asin', 'Image Urls', 'dec_count'],
      dtype='object')

In [153]:

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"

MIN_TOKENS = 150 # Any less than this, and we don't have enough useful content
MAX_TOKENS = 160 # Truncate after this many tokens. Then after adding in prompt text, we will get to around 180 tokens

MIN_CHARS = 300
CEILING_CHARS = MAX_TOKENS * 7

class Item:
    """
    An Item is a cleaned, curated datapoint of a Product with a Price
    """

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    PREFIX = "Price is"
    QUESTION = "How much does this cost to the nearest dollar?"
    REMOVALS = ['"Batteries Included?": "No"', '"Batteries Included?": "Yes"', '"Batteries Required?": "No"', '"Batteries Required?": "Yes"', "By Manufacturer", "Item", "Date First", "Package", ":", "Number of", "Best Sellers", "Number", "Product "]

    title: str
    price: float
    category: str
    token_count: int = 0
    details: Optional[str]
    prompt: Optional[str] = None
    include = False

    def __init__(self, data):
        self.title = data['Product Title']
        self.price = data['Mrp']
        self.parse(data)

    def scrub_details(self):
        """
        Clean up the details string by removing common text that doesn't add value
        """
        details = self.details
        for remove in self.REMOVALS:
            details = details.replace(remove, "")
        return details

    def scrub(self, stuff):
        """
        Clean up the provided text by removing unnecessary characters and whitespace
        Also remove words that are 7+ chars and contain numbers, as these are likely irrelevant product numbers
        """
        stuff = re.sub(r'[\[\]"{}【】\s]+', ' ', stuff).strip()
        stuff = stuff.replace(" ,", ",").replace(",,,",",").replace(",,",",")
        words = stuff.split(' ')
        select = [word for word in words if len(word)<7 or not any(char.isdigit() for char in word)]
        return " ".join(select)

    def parse(self, data):
        """
        Parse this datapoint and if it fits within the allowed Token range,
        then set include to True
        """
        # ADD : Code block to identify and remove the rows where product description is not available
        self.contents = data['Product Description']
        if self.contents:
            self.contents += '\n'
        if len(self.contents) > MIN_CHARS:
            self.contents = self.contents[:CEILING_CHARS]
            text = f"{self.scrub(self.title)}\n{self.scrub(self.contents)}"
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            if len(tokens) > MIN_TOKENS:
                tokens = tokens[:MAX_TOKENS]
                text = self.tokenizer.decode(tokens)
                self.make_prompt(text)
                self.include = True

    def make_prompt(self, text):
        """
        Set the prompt instance variable to be a prompt appropriate for training
        """
        self.prompt = f"{self.QUESTION}\n\n{text}\n\n"
        self.prompt += f"{self.PREFIX}{str(round(self.price))}.00"
        self.token_count = len(self.tokenizer.encode(self.prompt, add_special_tokens=False))

    def test_prompt(self):
        """
        Return a prompt suitable for testing, with the actual price removed
        """
        return self.prompt.split(self.PREFIX)[0] + self.PREFIX

    def __repr__(self):
        """
        Return a String version of this Item
        """
        return f" {self.QUESTION}\n{self.title}  \n   The description of the product is:   {self.contents} \n {self.PREFIX}= ${self.price}"



In [154]:
def parse(data):
    """
    Parse this datapoint and if it fits within the allowed Token range,
    then set include to True
    """
    contents = data['Product Description']
    return contents

In [155]:
final_df.shape

(27394, 16)

In [156]:
Item(final_df.iloc[27100])

 How much does this cost to the nearest dollar?
 Old Spice Anti-Perspirant 2.6oz Hawkridge Solid (2 Pack)   
   The description of the product is:   Old Spice Anti-Perspirant 2.6oz Hawkridge Solid (2 Pack)
 
 Price is= $4849000

In [157]:
final_df.shape[0]-1

27393

In [158]:
final_df.to_csv('final_df.csv')

We need to convert the Price column to float before executing this step

In [160]:
final_df['Mrp'] = final_df['Mrp'].astype('float64')

<ipython-input-160-63702fd71816>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['Mrp'] = final_df['Mrp'].astype('float64')


In [165]:
final_df.groupby('dec_count')['dec_count'].sum()

,dec_count
dec_count,
0,0


In [ ]:
final_df.shape[0]-1

In [172]:
items =[]
counter = 0
counter
for i in range(0, final_df.shape[0]-1):
  try:
    price= float(final_df.iloc[i]["Mrp"])
    if price > 0.0:
      item = Item(final_df.iloc[i])
      counter +=1
      #print(counter)
      #items.append(item)


      """ Below two lines are useful for the token limit to be reached."""
      if item.include:
        items.append(item)

  except ValueError as e:
    print("not included")
    pass
print(counter)

27393


In [173]:
len(items)

5689

In [176]:
import pandas as pd

def add_rows_to_dataframe(items):
  """
  Adds rows to a DataFrame with 'prompt' and 'price' columns
  based on a list of items in the format "prompt $price".

  Args:
    items: A list of strings, where each string is in the format "prompt $price".

  Returns:
    A pandas DataFrame with 'prompt' and 'price' columns.
  """

  fg = pd.DataFrame(columns=['prompt', 'price'])

  for item in items:
    try:
      prompt, price_str = str(item).split('$')
      price = float(price_str)
      fg = fg._append({'prompt': prompt, 'price': price}, ignore_index=True)
    except (ValueError, IndexError):
      print(f"Error processing item: {item}")

  return fg



final_df = add_rows_to_dataframe(items)

<ipython-input-176-19866b93355d>:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fg = fg._append({'prompt': prompt, 'price': price}, ignore_index=True)


Error processing item:  How much does this cost to the nearest dollar?
 Lip Balm Container Tubes - 50-Pack - DIY - Purple - 3/16 Oz (5.5 ml) - Including 50 Writeable (Purple) & 50 Printed Lip Balm Stickers - Twist Mechanism and a Cap - Empty - Make Natural Lip Balm   
   The description of the product is:   DIY LIP BALM CONTAINER KIT - This kit contains 50 Purple Lip Balm Container Tubes that can be used to make your own lip balm chapsticks. Each Lip Balm Container has a twist mechanism and a cap and can contain 3/16 Oz (5.5 ml) of lip balm. LOTS OF FUN - Making your own lip balm is a lot of fun! You can find many lip balm recipes online, with all sorts of natural, organic flavours and colors. Depending on the ingredients, lip balm can be used to moisturize and relieve chapped or dry lips in the winter or as a sunscreen in the summer. LABELS INCLUDED - Each of the 50 Lip Balm Container Tubes can be labelled. You can choose between 2 types of labels: writeable (purple) and printed. Use 

In [177]:
final_df.shape

(5683, 2)

In [179]:
final_df.to_csv('curated_dataset.csv')

In [34]:
fg = pd.DataFrame(columns=['prompt' , 'price'])

#fg.columns = ['prompt' , 'price']
d = {}
for i in range(0,len(items)):
  d['prompt']



Save a dataset as train and test in pickle format to use it later

In [180]:
import pandas as pd
from sklearn.model_selection import train_test_split
# Split data into train and test sets
train, test = train_test_split(final_df, test_size=0.2, random_state=42)

# Save train DataFrame to 'train.pkl'
train.to_pickle('train.pkl')

# Save test DataFrame to 'test.pkl'
test.to_pickle('test.pkl')